# Lecture 6: Prepare Forest-Fire Data for Regression Models

### Short, simple, self-study notes

EDA and cleaning are complete. This lesson prepares the Algerian Forest Fires data for Linear Regression, Ridge, Lasso, and Elastic Net.

**Main idea:** Split first, learn feature decisions from training data only, and keep the test data unseen until final evaluation.

## 1. Model-training roadmap

Before fitting a regression model, follow this safe order:

1. Load the cleaned dataset.
2. Encode text features such as fire/not fire.
3. Separate X features from the y target.
4. Split into training and test data.
5. Inspect multicollinearity using training data only.
6. Apply the same selected feature columns to training and test data.
7. Fit scaling on training data, then transform both sets.
8. Train and tune models in the next lesson.

For this project, **FWI** is y, the numeric value we want to predict.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data_path = Path("Ridge Lassso Elastic Regression Practicals") / "Algerian_forest_fires_cleaned_dataset.csv"
df = pd.read_csv(data_path)

df["Classes"] = df["Classes"].str.strip().str.lower()
df["Classes"] = df["Classes"].map({"not fire": 0, "fire": 1})

if df["Classes"].isna().any():
    raise ValueError("Unexpected class label found. Check the text cleaning step.")

df.head()

## 2. Create X and y, then split

- **X** is the table of independent features.
- **y** is the target value to predict.
- A test size of 0.25 reserves 25% of the data for the final test.
- A random state makes the split repeatable.

We remove day, month, and year for this first regression example. Time fields can be useful in another project, but here the lesson focuses on weather and fire-index features.

In [ ]:
features_to_remove = ["day", "month", "year", "FWI"]
X = df.drop(columns=features_to_remove)
y = df["FWI"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)
print("Training target:", y_train.shape)
print("Test target:", y_test.shape)
X_train.head()

## 3. Feature selection using correlation

Two input features that contain nearly the same information are highly correlated. This is called **multicollinearity**.

Why check it?

- It can make ordinary linear-regression coefficients unstable.
- It can make model explanations harder.
- It may add duplicate information without helping prediction.

We use the absolute correlation because both strong positive and strong negative relationships can indicate duplicate information. The threshold is a modelling choice, not a universal rule. Domain knowledge and cross-validation should guide the final decision.

In [ ]:
def highly_correlated_features(data, threshold=0.90):
    """Return one feature from each pair whose absolute correlation exceeds the threshold."""
    correlation = data.corr().abs()
    upper_triangle = correlation.where(
        np.triu(np.ones(correlation.shape), k=1).astype(bool)
    )
    return [column for column in upper_triangle.columns if (upper_triangle[column] > threshold).any()]

threshold = 0.90
features_to_drop = highly_correlated_features(X_train, threshold=threshold)

print("Features chosen for removal from the training correlation check:", features_to_drop)

X_train_reduced = X_train.drop(columns=features_to_drop)
X_test_reduced = X_test.drop(columns=features_to_drop)

print("Before selection:", X_train.shape, X_test.shape)
print("After selection:", X_train_reduced.shape, X_test_reduced.shape)

### Important rule: do not inspect test correlations

Feature selection is learned from X_train only. Then the same columns are removed from X_test.

If we use the test set to choose features, we leak test information into the training process. The final test score would then be too optimistic.

In [ ]:
plt.figure(figsize=(11, 8))
sns.heatmap(X_train.corr(), cmap="coolwarm", center=0, square=True, linewidths=0.3)
plt.title("Training-feature correlation heatmap", weight="bold")
plt.show()

## 4. Standardization: put features on a similar scale

Features use different units. For example, rain may be near 0 to 16, while some fire indexes can be much larger. Ridge, Lasso, and Elastic Net penalize coefficient sizes, so feature scale matters.

Standardization changes each feature approximately to:

**new value = (old value - training mean) / training standard deviation**

After scaling, each training feature has mean near 0 and standard deviation near 1.

**Correct method:** fit_transform on training data; transform on test data. The test set must not influence the training mean or standard deviation.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_reduced)
X_test_scaled = scaler.transform(X_test_reduced)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train_reduced.columns, index=X_train_reduced.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test_reduced.columns, index=X_test_reduced.index)

print("Average training-feature mean after scaling:", round(X_train_scaled.mean().mean(), 6))
print("Average training-feature standard deviation after scaling:", round(X_train_scaled.std(ddof=0).mean(), 6))
X_train_scaled.head()

## 5. Visual: before and after standardization

The left box plot shows different original units. The right box plot shows the same features after standardization. Scaling changes the units, but it does not remove real outliers or create new information.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.boxplot(data=X_train_reduced, ax=axes[0], color="#9ecae1")
axes[0].set_title("Before standardization")
axes[0].tick_params(axis="x", rotation=70)

sns.boxplot(data=X_train_scaled, ax=axes[1], color="#a1d99b")
axes[1].set_title("After standardization")
axes[1].tick_params(axis="x", rotation=70)

fig.suptitle("Standardization aligns feature scales", weight="bold")
fig.tight_layout()
plt.show()

## 6. Final revision card

- Encode text categories only after cleaning their spaces and spelling.
- Set X as input features and y as the FWI target.
- Split into training and test data before correlation checks or scaling.
- Use training data only to decide which correlated features to remove.
- Drop the same selected columns from X_train and X_test.
- Standardize for Ridge, Lasso, and Elastic Net because their penalties depend on coefficient size.
- Fit the scaler on training data only; transform the test data with that same scaler.
- Next step: fit Linear Regression, Ridge, Lasso, and Elastic Net, then compare test scores.

### One-line interview answer

**To avoid data leakage, I split the data first, learn feature selection and scaling from the training set only, and apply the same learned steps to the test set.**